# Session 1 Suggested Exercises - Interview Training Notebook

Goal: build hands-on fluency for quant researcher interviews by practicing the exact topics from Session 1 in a structured order.

## How To Use This Notebook

1. Run cells in order.
2. Do not skip reflection prompts; they are interview rehearsal.
3. For each section, capture: speed, correctness, trade-offs, and production considerations.
4. Repeat weak sections until you can explain them without notes.

In [1]:
# Progress tracker
progress = {
    'env_git_docker': False,
    'timing_decorator': False,
    'pandas_benchmark': False,
    'binary_storage': False,
    'parallel_multiprocessing': False,
    'parallel_multithreading': False,
    'parallel_ray_local': False,
    'parallel_ray_remote': False,
    'scheduler_dag': False,
    'interview_drill': False,
}
progress

{'env_git_docker': False,
 'timing_decorator': False,
 'pandas_benchmark': False,
 'binary_storage': False,
 'parallel_multiprocessing': False,
 'parallel_multithreading': False,
 'parallel_ray_local': False,
 'parallel_ray_remote': False,
 'scheduler_dag': False,
 'interview_drill': False}

## Section 1: Environment, Git, Docker (Concept + Practice)

### Exercise
- Create a GitHub repo named `alpha-practice`.
- Write a `Dockerfile` that installs Miniconda and creates a Python environment.
- Build and run the image.

### Interview prompts
- Why does environment reproducibility matter in research?
- Difference between image and container?
- Why Git branching strategy matters for research velocity?

Mark `progress['env_git_docker'] = True` after completion.

In [2]:
# Section 1 evidence: reproducibility, image vs container, git workflow
from pathlib import Path
import subprocess
import hashlib
import json

def run_cmd(cmd):
    try:
        out = subprocess.run(cmd, capture_output=True, text=True, check=True)
        text = out.stdout.strip() or out.stderr.strip()
        return text if text else '<no output>'
    except Exception as e:
        return f'Unavailable: {e}'

dockerfile_candidates = [Path('Dockerfile'), Path('./alpha-session1-practice/Dockerfile')]
dockerfile_path = next((p for p in dockerfile_candidates if p.exists()), None)
docker_sha = hashlib.sha256(dockerfile_path.read_bytes()).hexdigest()[:16] if dockerfile_path else None

commands = {
    'git_version': run_cmd(['git', '--version']),
    'git_remote': run_cmd(['git', 'remote', '-v']) if Path('.git').exists() else 'not a git repo in cwd',
    'git_branch': run_cmd(['git', 'branch', '--show-current']) if Path('.git').exists() else 'not a git repo in cwd',
    'docker_version': run_cmd(['docker', '--version']),
    'docker_image': run_cmd(['docker', 'image', 'ls', 'alpha-session1:latest', '--format', '{{.Repository}}:{{.Tag}} {{.ID}}']),
    'docker_run_python': run_cmd(['docker', 'run', '--rm', 'alpha-session1:latest', 'bash', '-c', 'python --version && conda list | head -n 5']),
    'dockerfile_found': str(dockerfile_path) if dockerfile_path else 'not found',
    'dockerfile_hash_prefix': docker_sha,
}

print(json.dumps(commands, indent=2))

section1_answer = {
    'Q1_reproducibility': 'Pinning dependencies and image hash makes research runs repeatable and auditable.',
    'Q2_image_vs_container': 'Image is immutable template; container is runtime instance with state.',
    'Q3_git_branching_velocity': 'Feature branches isolate experiments, reduce merge risk, and preserve research iteration speed.'
}
section1_answer

{
  "git_version": "git version 2.32.1 (Apple Git-133)",
  "git_remote": "origin\tgit@github.com:Hunghotin/alpha-session1-practice.git (fetch)\norigin\tgit@github.com:Hunghotin/alpha-session1-practice.git (push)",
  "git_branch": "main",
  "docker_version": "Docker version 26.0.0, build 2ae903e",
  "docker_image": "alpha-session1:latest 5e5616fa0dd6",
  "docker_run_python": "Python 3.11.15\n# packages in environment at /opt/miniconda:\n#\n# Name                      Version          Build               Channel\n_libgcc_mutex               0.1              main\n_openmp_mutex               5.1              51_gnu",
  "dockerfile_found": "Dockerfile",
  "dockerfile_hash_prefix": "1272ec913429cc22"
}


{'Q1_reproducibility': 'Pinning dependencies and image hash makes research runs repeatable and auditable.',
 'Q2_image_vs_container': 'Image is immutable template; container is runtime instance with state.',
 'Q3_git_branching_velocity': 'Feature branches isolate experiments, reduce merge risk, and preserve research iteration speed.'}

### Task Completion Notes

#### Implementation Steps
1. Started from a base image that is reachable in the current environment.
2. Installed system tools needed for research workflows, including `wget`, `curl`, `git`, and build tools.
3. Installed Miniconda inside the image.
4. Created a Python environment with the required scientific packages.
5. Exposed the environment through `PATH` so the container works cleanly with non-interactive commands.
6. Kept the notebook evidence cell focused on reproducibility checks and interview-ready answers.

#### Bug Points to Watch
- Image builds depend on external registry access; a reachable base image source is required.
- CPU architecture must match the Miniconda installer and the Docker platform.
- Non-interactive `docker run ... bash -c` commands should not rely on `conda activate` unless shell initialization is handled explicitly.
- Package installation can be blocked by channel policy prompts, so channel selection should be explicit.
- Terminal comments starting with `#` should not be pasted as commands.

#### Current Outcome
- The Docker image is ready for use in the notebook workflow.
- The Section 1 exercise now has a clear, reproducible environment story for interview practice.


In [3]:
import time
from functools import wraps

def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # TODO 1: 记录开始时间 (提示: 使用 time.perf_counter())
        # TODO 2: 运行目标函数 func(*args, **kwargs) 并保存结果
        # TODO 3: 记录结束时间，计算 elapsed 耗时，并打印出来
        # TODO 4: 返回函数结果
        start = time.perf_counter()
        result = func(*args, **kwargs)
        end = time.perf_counter()
        print(f"[timeit] {func.__name__} : {end - start:.6f} seconds")
        return result
    return wrapper

@timed
def sample_work(n=1_000_000):
    s = 0
    for i in range(n):
        s += i
    return s

print("Testing Timing Decorator")
result = sample_work(200_000)
print(f"结果: {result}")
assert result == 19999900000

Testing Timing Decorator
[timeit] sample_work : 0.004287 seconds
结果: 19999900000


In [4]:
import time
from functools import wraps

@timed
def sample_work(n=1_000_000):
    s = 0
    for i in range(n):
        s += i
    return s

sample_work(200_000)

[timeit] sample_work : 0.005390 seconds


19999900000

## Section 2: Timing & Benchmarking

Exercise:
- Implement the `timed` decorator using `time.perf_counter()` to measure runtime and print elapsed time.
- Measure timer resolution and benchmark variability with repeated runs, warm-up, and summary statistics.

Interview prompts:
- Why use `time.perf_counter()` instead of `time.time()`?
- What can make micro-benchmarks misleading?
- How would you design a fair and repeatable benchmark (input, warm-up, repeats, summary stats)?

In [5]:
# Section 2 evidence: timer quality + benchmark reliability
import statistics
import time

def timer_resolution(timer_fn, n=50000):
    vals = []
    last = timer_fn()
    for _ in range(n):
        now = timer_fn()
        delta = now - last
        if delta > 0:
            vals.append(delta)
        last = now
    return {
        'samples': len(vals),
        'min_positive_delta_us': min(vals) * 1e6 if vals else None,
        'median_delta_us': statistics.median(vals) * 1e6 if vals else None,
    }

perf_stats = timer_resolution(time.perf_counter)
wall_stats = timer_resolution(time.time)
print('perf_counter stats:', perf_stats)
print('time.time stats:', wall_stats)

def micro_bench(fn, repeats=8):
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return {
        'mean_ms': statistics.mean(times) * 1e3,
        'std_ms': statistics.stdev(times) * 1e3 if len(times) > 1 else 0.0,
        'raw_ms': [round(x * 1e3, 3) for x in times],
    }

bench_noise = micro_bench(lambda: sample_work(120_000), repeats=10)
print('benchmark variability (ms):', bench_noise)

section2_answer_card = {
    'Q1_perf_counter_vs_time': 'perf_counter usually provides higher-resolution monotonic timing for benchmarking.',
    'Q2_micro_benchmark_risk': 'Run-to-run variance exists; use repeats, warm-up, and robust summary stats.',
    'Q3_fair_benchmarking': 'Use identical input, same machine state, multiple repeats, and compare mean/std together.'
}
section2_answer_card

perf_counter stats: {'samples': 49999, 'min_positive_delta_us': 0.040978193283081055, 'median_delta_us': 0.08288770914077759}
time.time stats: {'samples': 3515, 'min_positive_delta_us': 0.7152557373046875, 'median_delta_us': 0.95367431640625}
[timeit] sample_work : 0.002782 seconds
[timeit] sample_work : 0.002917 seconds
[timeit] sample_work : 0.003135 seconds
[timeit] sample_work : 0.002717 seconds
[timeit] sample_work : 0.002575 seconds
[timeit] sample_work : 0.002574 seconds
[timeit] sample_work : 0.002497 seconds
[timeit] sample_work : 0.002598 seconds
[timeit] sample_work : 0.002731 seconds
[timeit] sample_work : 0.002392 seconds
benchmark variability (ms): {'mean_ms': 2.715675113722682, 'std_ms': 0.21827735895443068, 'raw_ms': [2.792, 2.939, 3.147, 2.734, 2.581, 2.58, 2.5, 2.744, 2.744, 2.396]}


{'Q1_perf_counter_vs_time': 'perf_counter usually provides higher-resolution monotonic timing for benchmarking.',
 'Q2_micro_benchmark_risk': 'Run-to-run variance exists; use repeats, warm-up, and robust summary stats.',
 'Q3_fair_benchmarking': 'Use identical input, same machine state, multiple repeats, and compare mean/std together.'}

## Section 3: Pandas — Missing Data & Reshaping

Exercise:
- Use `df_sparse` to forward-fill each `symbol` with multiple approaches: a Python loop, `groupby` + `ffill`, and `unstack()` → `ffill()` → `stack()`.
- Benchmark each method for runtime, memory usage, and correctness, then summarize the results in a comparison table.

Interview prompts:
- When can `unstack` + `ffill` be faster than `groupby`?
- What is the memory trade-off between wide and long formats?
- If the dataset is large, which method should you prioritize to balance speed and memory?

In [6]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
dates = pd.date_range('2024-01-01', periods=730, freq='B')
symbols = [f'STK{i:03d}' for i in range(60)]
idx = pd.MultiIndex.from_product([dates, symbols], names=['date', 'symbol'])
df = pd.DataFrame({'value': rng.normal(size=len(idx))}, index=idx).sort_index()

drop_n = int(0.1 * len(df))
to_drop = rng.choice(len(df), size=drop_n, replace=False)
df_sparse = df.drop(df.index[to_drop]).sort_index()
df_sparse = df_sparse.reindex(df.index)

df.shape, df_sparse.shape

((43800, 1), (43800, 1))

In [7]:
import time

def bench(name, fn):
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    return {'method': name, 'seconds': dt, 'rows': len(out)}

def ffill_loop():
    out = df_sparse.copy()
    for sym in out.index.get_level_values('symbol').unique():
        m = out.index.get_level_values('symbol') == sym
        out.loc[m, 'value'] = out.loc[m, 'value'].ffill()
    return out

def ffill_apply():
    out = df_sparse.copy()
    out['value'] = out.groupby(level='symbol')['value'].apply(lambda x: x.ffill()).droplevel(0)
    return out

def ffill_groupby():
    out = df_sparse.copy()
    out['value'] = out.groupby(level='symbol')['value'].ffill()
    return out

def ffill_unstack_stack():
    wide = df_sparse['value'].unstack('symbol').copy()
    wide = wide.ffill()
    #out = wide.stack(dropna=False).to_frame('value').sort_index()
    out = wide.stack().to_frame('value').sort_index()
    return out

results = [
    bench('loop', ffill_loop),
    bench('apply', ffill_apply),
    bench('groupby', ffill_groupby),
    bench('unstack_ffill_stack', ffill_unstack_stack),
]
pd.DataFrame(results).sort_values('seconds')

,method,seconds,rows
2,groupby,0.001900,43800
3,unstack_ffill_stack,0.004272,43800
1,apply,0.006790,43800
0,loop,0.111997,43800


In [8]:
from pathlib import Path

base = Path('session1_artifacts')
single_dir = base / 'single'
date_dir = base / 'shard_by_date'
symbol_dir = base / 'shard_by_symbol'
for d in [single_dir, date_dir, symbol_dir]:
    d.mkdir(parents=True, exist_ok=True)

single_path = single_dir / 'panel.parquet'
parquet_available = True

try:
    # TODO 1: 将 df_sparse 存为 parquet 格式到 single_path 路径
    df_sparse.to_parquet(single_path)
    pass
except Exception as e:
    parquet_available = False
    single_path = single_dir / 'panel.pkl'
    df_sparse.to_pickle(single_path)
    print('Parquet unavailable, fallback to pickle:', e)

print("检查单文件是否生成:", single_path.exists())
single_path, parquet_available

检查单文件是否生成: True


(PosixPath('session1_artifacts/single/panel.parquet'), True)

In [9]:
# Shard by date and symbol
if parquet_available:
    # TODO 2: 按 'date' 维度 groupby 遍历 df_sparse
    # 提示: name格式 => date_dir / f'date={dt.date()}.parquet'
    # 对每个切片单独存为 parquet
    dfs_grouped_date = df_sparse.groupby(level='date')
    for dt, g in dfs_grouped_date:
        name = date_dir / f'date={dt.date()}.parquet'
        g.to_parquet(name)

    # TODO 3: 按 'symbol' 维度 groupby 遍历 df_sparse
    # 提示: name格式 => symbol_dir / f'symbol={sym}.parquet'
    # 对每个切片单独存为 parquet
    dfs_grouped_symbol = df_sparse.groupby(level='symbol')
    for sym, g in dfs_grouped_symbol:
        name = symbol_dir / f'symbol={sym}.parquet'
        g.to_parquet(name)

    shard_counts = (len(list(date_dir.glob('*.parquet'))), len(list(symbol_dir.glob('*.parquet'))))
else:
    for dt, g in df_sparse.groupby(level='date'):
        out = date_dir / f'date={dt.date()}.pkl'
        g.to_pickle(out)

    for sym, g in df_sparse.groupby(level='symbol'):
        out = symbol_dir / f'symbol={sym}.pkl'
        g.to_pickle(out)

    shard_counts = (len(list(date_dir.glob('*.pkl'))), len(list(symbol_dir.glob('*.pkl'))))

print("Check:", shard_counts)
shard_counts

Check: (730, 60)


(730, 60)

## Section 4: Binary Storage (single file + sharding)

In [10]:
from pathlib import Path
import pyarrow

base = Path('./session1_artifacts')
single_dir = base / 'single'
date_dir = base / 'shard_by_date'
symbol_dir = base / 'shard_by_symbol'
for d in [single_dir, date_dir, symbol_dir]:
    d.mkdir(parents=True, exist_ok=True)

single_path = single_dir / 'panel.parquet'
parquet_available = True
try:
    df_sparse.to_parquet(single_path)
    parquet_roundtrip = pd.read_parquet(single_path)
    print('Parquet engine:', pyarrow.__version__)
    print('Parquet file:', single_path)
    print('Parquet rows/cols:', parquet_roundtrip.shape)
except Exception as e:
    parquet_available = False
    single_path = single_dir / 'panel.pkl'
    df_sparse.to_pickle(single_path)
    print('Parquet unavailable, fallback to pickle:', e)

single_path, parquet_available

Parquet engine: 23.0.1
Parquet file: session1_artifacts/single/panel.parquet
Parquet rows/cols: (43800, 1)


(PosixPath('session1_artifacts/single/panel.parquet'), True)

In [11]:
# Prepare a day-wise input list
daily_inputs = [g for _, g in df_sparse.groupby(level='date')]

def daily_avg(day_df):
    return float(day_df['value'].mean())

# Sequential baseline
t0 = time.perf_counter()
# TODO 1: 使用列表推导式 (list comprehension) 将 daily_avg 应用到 daily_inputs 每一个元素上
baseline = [daily_avg(day_df) for day_df in daily_inputs] 
baseline_seconds = time.perf_counter() - t0
print(f'sequential baseline took {baseline_seconds:.6f}s')

print("check:", len(baseline))
baseline[:3]

sequential baseline took 0.020726s
check: 730


[0.10181700933164212, -0.21632867850823517, -0.042687950014410135]

In [12]:
from multiprocessing import Pool, cpu_count
import numpy as np
import time

workers = max(1, cpu_count() - 1)
daily_values = [day_df['value'].to_numpy() for day_df in daily_inputs]
try:
    t0 = time.perf_counter()
    # TODO 2: Use multiprocessing.Pool with a picklable built-in reducer
    with Pool(workers) as p:
        mp_out = p.map(np.mean, daily_values)
    mp_seconds = time.perf_counter() - t0
    print(f'multiprocessing took {mp_seconds:.6f}s')
except Exception as e:
    mp_out = baseline[:]
    mp_seconds = float('nan')
    print('multiprocessing unavailable, fallback to baseline:', e)

print("check:", len(mp_out) == len(daily_inputs))
mp_out[:3]

multiprocessing took 0.464995s
check: True


[np.float64(nan), np.float64(nan), np.float64(nan)]

In [13]:
from concurrent.futures import ThreadPoolExecutor
import time

t0 = time.perf_counter()
# TODO 3: 使用 ThreadPoolExecutor 上下文管理器, 并调用 ex.map 映射 daily_avg 到 daily_inputs
with ThreadPoolExecutor(max_workers=workers) as ex:
    th_out = list(ex.map(daily_avg, daily_inputs))  

th_seconds = time.perf_counter() - t0
print(f'multithreading took {th_seconds:.6f}s')

print("Check:", len(th_out) == len(daily_inputs))
th_out[:3]

multithreading took 0.053042s
Check: True


[0.10181700933164212, -0.21632867850823517, -0.042687950014410135]

## Section 5: Parallelization - Daily Average

Core task from suggested exercise: compute daily average and parallelize by day.

In [14]:
# Ray local cluster (optional if ray is installed)
try:
    import ray

    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True)

    # TODO 4: use ray.remote decorator to decorate daily_avg function, and create daily_avg_remote version
    @ray.remote
    def daily_avg_remote(day_df):
        return float(day_df['value'].mean())
    
    t0 = time.perf_counter()
    # TODO 5: use remote_func.remote() submit tasks to get future list,and ray.get() to collect results
    ray_out = []
    for day_df in df_sparse.groupby(level='date'):
        ray_out.append(daily_avg_remote.remote(day_df[1]))
    ray_out = ray.get(ray_out)
    ray_seconds = time.perf_counter() - t0
    print(f'ray local took {ray_seconds:.6f}s')
    print('ray local ok', len(ray_out), ray_out[:3])
except Exception as e:
    print('Ray local skipped:', e)

2026-05-10 20:55:01,623	INFO worker.py:1814 -- Connecting to existing Ray cluster at address: 127.0.0.1:6379...
2026-05-10 20:55:01,635	INFO worker.py:2012 -- Connected to Ray cluster.
/Users/huanghaotian/anaconda3/envs/course311/lib/python3.11/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


ray local took 0.801806s
ray local ok 730 [0.10181700933164212, -0.21632867850823517, -0.042687950014410135]


In [15]:
from multiprocessing import Pool, cpu_count
import numpy as np
import time

workers = max(1, cpu_count() - 1)
daily_values = [day_df['value'].to_numpy() for day_df in daily_inputs]
try:
    t0 = time.perf_counter()
    with Pool(processes=workers) as p:
        mp_out = p.map(np.mean, daily_values)
    mp_seconds = time.perf_counter() - t0
    print(f'multiprocessing took {mp_seconds:.6f}s')
except Exception as e:
    mp_out = baseline[:]
    mp_seconds = float('nan')
    print('multiprocessing unavailable, fallback to baseline:', e)

len(mp_out), mp_out[:3]

multiprocessing took 0.465714s


(730, [np.float64(nan), np.float64(nan), np.float64(nan)])

In [16]:
from concurrent.futures import ThreadPoolExecutor
import time

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=workers) as ex:
    th_out = list(ex.map(daily_avg, daily_inputs))
th_seconds = time.perf_counter() - t0
print(f'multithreading took {th_seconds:.6f}s')

len(th_out), th_out[:3]

multithreading took 0.026780s


(730, [0.10181700933164212, -0.21632867850823517, -0.042687950014410135])

### Interview prompts
- Why might multiprocessing beat multithreading for CPU-bound tasks in Python?
- What overhead can erase multiprocessing gains?
- What changes if the task becomes IO-bound?

In [17]:
# Section 5 evidence: CPU-bound vs IO-bound behavior
import math

# Correctness check
print('baseline == multiprocessing:', np.allclose(baseline, mp_out))
print('baseline == multithreading:', np.allclose(baseline, th_out))

speed_table = pd.DataFrame([
    {'method': 'sequential', 'seconds': baseline_seconds},
    {'method': 'multiprocessing', 'seconds': mp_seconds},
    {'method': 'multithreading', 'seconds': th_seconds},
]).sort_values('seconds').reset_index(drop=True)
speed_table['speedup_vs_seq'] = baseline_seconds / speed_table['seconds']
print(speed_table)

# IO-bound toy task to show thread usefulness
def io_task(x):
    time.sleep(0.01)
    return x

io_inputs = list(range(120))
t0 = time.perf_counter()
_ = [io_task(x) for x in io_inputs]
io_seq = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=workers) as ex:
    _ = list(ex.map(io_task, io_inputs))
io_th = time.perf_counter() - t0

print({'io_seq_sec': round(io_seq, 4), 'io_thread_sec': round(io_th, 4)})

section5_answer_card = {
    'Q1_cpu_bound_mp_vs_mt': 'CPU-bound tasks often favor multiprocessing due to GIL constraints.',
    'Q2_mp_overhead': 'Process spawn/serialization/scheduling overhead can erase gains for small tasks.',
    'Q3_io_bound_change': 'For IO-bound workloads, threads can outperform sequential due to wait overlap.'
}
section5_answer_card

baseline == multiprocessing: False
baseline == multithreading: True
            method   seconds  speedup_vs_seq
0       sequential  0.020726        1.000000
1   multithreading  0.026780        0.773945
2  multiprocessing  0.465714        0.044504


(raylet) [2026-05-10 20:55:03,692 E 55053 24388270] (raylet) file_system_monitor.cc:116: /tmp/ray/session_2026-05-10_20-54-22_571170_55047 is over 95% full, available space: 20.4786 GB; capacity: 460.432 GB. Object creation will fail if spilling is required.


{'io_seq_sec': 1.4193, 'io_thread_sec': 0.2042}


{'Q1_cpu_bound_mp_vs_mt': 'CPU-bound tasks often favor multiprocessing due to GIL constraints.',
 'Q2_mp_overhead': 'Process spawn/serialization/scheduling overhead can erase gains for small tasks.',
 'Q3_io_bound_change': 'For IO-bound workloads, threads can outperform sequential due to wait overlap.'}

In [18]:
# Ray local cluster (optional if ray is installed)
try:
    import ray

    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True)

    @ray.remote
    def daily_avg_remote(day_df):
        return float(day_df['value'].mean())

    t0 = time.perf_counter()
    ray_out = ray.get([daily_avg_remote.remote(x) for x in daily_inputs])
    ray_seconds = time.perf_counter() - t0
    print(f'ray local took {ray_seconds:.6f}s')
    print('ray local ok', len(ray_out), ray_out[:3])
except Exception as e:
    print('Ray local skipped:', e)

ray local took 0.367527s
ray local ok 730 [0.10181700933164212, -0.21632867850823517, -0.042687950014410135]


### Remote Ray cluster template

Use this when you have a remote head node: `ray.init(address='ray://<host>:10001')`

Interview prompt: local multiprocessing vs distributed ray, compare scheduling overhead, fault tolerance, and scaling limits.

In [19]:
import ray
import time
import numpy as np

# Disconnect any existing local ray session
if ray.is_initialized():
    ray.shutdown()

print("Connecting to remote Ray cluster...")

try:
    # Connect to the remote Ray cluster using localhost (replace with actual remote IP if needed)
    # Ensure you start a ray head locally first by running in your terminal:
    # ray start --head --port=6379 --ray-client-server-port=10001
    ray.init(address='ray://localhost:10001', ignore_reinit_error=True)
    
    @ray.remote
    def daily_avg_remote_cluster(day_data):
        return np.mean(day_data)
        
    if 'daily_inputs' in locals():
        t0 = time.time()
        
        # Dispatch tasks to remote ray cluster
        futures = [daily_avg_remote_cluster.remote(d) for d in daily_inputs]
        remote_ray_out = ray.get(futures)
        
        remote_ray_seconds = time.time() - t0
        
        print(f"Computed averages for {len(remote_ray_out)} days using remote Ray cluster.")
        print(f"Time taken: {remote_ray_seconds:.4f} seconds")
        
        # Record stats
        if 'perf_stats' in locals():
            perf_stats['Remote Ray'] = remote_ray_seconds
    else:
        print("Missing 'daily_inputs', please run the parallel generation cells above first.")
        
except Exception as e:
    print(f"Could not connect to remote Ray cluster.\nError: {e}")
    print("\nPlease start the cluster before running: \nray start --head --port=6379 --ray-client-server-port=10001")
finally:
    if ray.is_initialized():
        ray.shutdown()

section5b_answer_card = {
    'Q4_local_vs_remote_ray': 'Remote Ray improves horizontal scalability and fault isolation, but adds network and scheduling overhead.'
}
section5b_answer_card

2026-05-10 20:55:05,333	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: ignore_reinit_error, log_to_driver


Connecting to remote Ray cluster...


SIGTERM handler is not set because current thread is not the main thread.


Computed averages for 730 days using remote Ray cluster.
Time taken: 1.7848 seconds


{'Q4_local_vs_remote_ray': 'Remote Ray improves horizontal scalability and fault isolation, but adds network and scheduling overhead.'}

## Section 6: Scheduling Dependent Jobs

Goal: create a chain of simple jobs where each depends on previous completion.

In [20]:
# Simple in-notebook dependency chain (conceptual DAG)
import time

state = {}

def job_extract():
    state['raw'] = [1, 2, 3, 4]
    print('extract done')

def job_transform():
    if 'raw' not in state:
        raise RuntimeError('extract must run first')
    state['features'] = [x * 10 for x in state['raw']]
    print('transform done')

def job_load():
    if 'features' not in state:
        raise RuntimeError('transform must run first')
    state['loaded'] = True
    print('load done')

job_extract()
job_transform()
job_load()
state

extract done
transform done
load done


{'raw': [1, 2, 3, 4], 'features': [10, 20, 30, 40], 'loaded': True}

### Production extension
- Rebuild this flow in Airflow or Dagster as a real DAG.
- Add retries, alerting, and idempotency.

Interview prompts
- What does idempotent job design mean?
- How do you prevent partial writes from corrupting downstream jobs?

In [28]:
# DAG implementation using Airflow TaskFlow API (Conceptual for Notebook)
# In production, this would be saved as a .py file in the Airflow DAGs folder.
try:
    from airflow.decorators import dag, task
    from datetime import datetime, timedelta

    # @dag defines the workflow graph
    # Common Parameters:
    # - start_date: The execution start date for the DAG (required), usually a fixed date in the past.
    # - schedule: The scheduling interval, e.g., '20 21 * * *' (9:20 PM daily), '@daily', or None.
    # - catchup: Set to False to prevent Airflow from running missed historic intervals since start_date.
    # - tags: Helps categorize and filter DAGs in the Airflow UI.
    # - default_args: Default parameters passed to all downstream tasks, e.g., timeouts and retries.
    @dag(
        start_date=datetime(2024, 1, 1), 
        schedule='20 21 * * *',  # Scheduled to run at 21:20 every night 
        catchup=False,
        tags=['quant', 'research'],
        default_args={
            'retries': 3,                           # Number of automatic retries on failure
            'retry_delay': timedelta(minutes=5),    # Cool-down wait time between retries
            'execution_timeout': timedelta(hours=2) # Max task execution time (prevents deadlocks)
        }
    )
    def my_research_pipeline():

        @task
        def extract():
            print("Extracting raw data...")
            return [1, 2, 3, 4]

        @task
        def transform(raw_data):
            print("Transforming to features...")
            return [x * 10 for x in raw_data]

        @task
        def load(feature_data):
            print(f"Loading features: {feature_data}")
            return True

        # Pipeline definition: DAG relies on explicitly passing outputs to inputs
        raw = extract()
        features = transform(raw)
        load(features)

    build_dag = my_research_pipeline()
    print("Airflow DAG defined successfully.")
except ImportError:
    print("Airflow is not installed in this environment. The example is conceptual.")

# ------------------------------------------------------------- #
# Alternative: Dagster Software-Defined Assets (SDA) Approach   #
# ------------------------------------------------------------- #
try:
    from dagster import (
        asset, 
        materialize, 
        RetryPolicy, 
        define_asset_job, 
        ScheduleDefinition,
        Definitions
    )

    # In Dagster, retry parameter configurations are typically attached at the asset level
    @asset(
        description="Extract raw market data",
        retry_policy=RetryPolicy(max_retries=3, delay=60) # Failure retry: up to 3 retries, 60s delay
    )
    def raw_data():
        return [1, 2, 3, 4]

    @asset
    def features(raw_data):
        return [x * 10 for x in raw_data]

    @asset
    def load_status(features):
        print(f"Dagster loaded features: {features}")
        return True

    # Define a job that targets all assets
    my_research_job = define_asset_job("my_research_job", selection="*")

    # Schedule the job to run at exactly 21:20 every night using a cron string
    my_daily_schedule = ScheduleDefinition(
        job=my_research_job,
        cron_schedule="21 21 * * *",
        execution_timezone="Asia/Shanghai" # Standard IANA timezone string
    )

    # --- PRODUCTION GLUE ---
    # In production, definitions are loaded by the Dagster daemon.
    # The daemon runs continuously in the background, reading this 'defs' object, 
    # monitoring the clock, and automatically executing the job when the schedule hits.
    defs = Definitions(
        assets=[raw_data, features, load_status],
        jobs=[my_research_job],
        schedules=[my_daily_schedule]
    )

    # --- LOCAL TEST ---
    # Can be tested locally in the notebook manually (bypassing the schedule daemon)
    result = materialize([raw_data, features, load_status])
    print("Dagster DAG materialized successfully:", result.success)
    
except ImportError:
    print("Dagster is not installed in this environment. The example is conceptual.")

/Users/huanghaotian/anaconda3/envs/course311/lib/python3.11/site-packages/dagster/_core/definitions/antlr_asset_selection/antlr_asset_selection.py:97: BetaWarning: Parameter `include_sources` of function `AssetSelection.all` is currently in beta, and may have breaking changes in minor version releases, with behavior changes in patch releases.
  return AssetSelection.all(include_sources=self.include_sources)
2026-05-10 21:24:20 +0800 - dagster - DEBUG - __ephemeral_asset_job__ - db514cd8-5448-48c0-8e8b-cba6a17f0df0 - 55580 - RUN_START - Started execution of run for "__ephemeral_asset_job__".
2026-05-10 21:24:20 +0800 - dagster - DEBUG - __ephemeral_asset_job__ - db514cd8-5448-48c0-8e8b-cba6a17f0df0 - 55580 - ENGINE_EVENT - Executing steps in process (pid: 55580)
2026-05-10 21:24:20 +0800 - dagster - DEBUG - __ephemeral_asset_job__ - db514cd8-5448-48c0-8e8b-cba6a17f0df0 - 55580 - RESOURCE_INIT_STARTED - Starting initialization of resources [io_manager].
2026-05-10 21:24:20 +0800 - dagste

Airflow is not installed in this environment. The example is conceptual.
Dagster loaded features: [10, 20, 30, 40]
Dagster DAG materialized successfully: True


In [38]:
import json
import subprocess
from pathlib import Path

proc = subprocess.run(
    ['bash', '-lc', 'chmod +x dag/run_daily.sh dag/install_cron.sh && ./dag/run_daily.sh'],
    capture_output=True,
    text=True,
)

out_path = Path('dag/run_job.out.json')
if out_path.exists():
    lines = [line.strip() for line in out_path.read_text().splitlines() if line.strip()]
    payload = json.loads(lines[-1]) if lines else {}
    print({'success': bool(payload.get('success')), 'returncode': proc.returncode})
else:
    print({'success': False, 'returncode': proc.returncode, 'error': 'dag/run_job.out.json missing'})

{'success': True, 'returncode': 0}


## Section 7: Interview Drill

Answer each in 60-90 seconds out loud:
1. Explain GIL impact on multiprocessing vs multithreading.
2. Explain your benchmark methodology and how you avoided bias.
3. Explain when to shard by date vs symbol.
4. Explain why reproducible environments are alpha-protective.
5. Explain a production-safe scheduled research pipeline.